<a href="https://colab.research.google.com/github/rzourob/Animation/blob/main/ICT_10_2026_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# تحليل سجل أعمال شعبة تكنولوجيا المعلومات – أكتوبر 2026
الدفتر يقرأ الملف من Google Drive، ينظّف البيانات، ثم يحللها حسب التصنيف والقسم والمكان والوقت والقطع المستخدمة.

**قبل التشغيل:** عدّل المتغير `FILE_PATH` في الخلية الثانية ليطابق مكان الملف في Drive.

## 1) ربط Google Drive وتجهيز المكتبات

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import numpy as np
import plotly.express as px
import plotly.io as pio
pio.templates.default = "plotly_white"
pd.set_option('display.max_colwidth', 120)

Mounted at /content/drive


## 2) قراءة الملف

In [3]:
# عدّل المسار حسب مكان الملف في Drive
FILE_PATH = '/content/drive/MyDrive/testRami/ICT 10-2026.csv'

raw = pd.read_csv(FILE_PATH, encoding='utf-8-sig')
print(raw.shape)
raw.head()

(50, 8)


,#,التاريخ,المكان,القسم,العطل/الطلب,الإجراء,القطع المستخدمة,التصنيف الرئيسي
0,1,30/9/2026,وحدة تاهيل سمعي,شعبة السمع والتوازن,ضبط عدادات البريد الاكتروني للزملاء في وحدة التشخيص - قسم السمع والتوازن,اعاده تويد باسورد جديدة,لا يوجد,الدعم الفني والمستخـدمين
1,2,1/10/2026,مكتب ICT,شعبة تكنولوجيا المعلومات,التنسيق مع وزارة الصحة بشئن امكانية مشاهدة IP البصمة لتجربة برنامج الموارد البشرية,التنسيق,لا يوجد,إدارة الخوادم وأمن المعلومات
2,3,1/10/2026,خلف غرفة الاجتماعات المركزية,قسم الخدمات المساندة,صيانة كاميرة المراقبة بجوار غرفة الاجتماعات وتغطي الساحة الخارجية للمستشفي,تغير Female rj45,Female rj45,الشبكات والاتصالات
3,4,4/10/2026,NaN,NaN,اجازة,NaN,NaN,NaN
4,5,5/10/2026,صاله انتظار عيادات الخارجية,قسم الخدمات المساندة,تمديد كابل كاميرا جديد بدل الكابل القديم المتهالك cam-07 /DVR-03,تمديد كابل كاميرات,كابل كاميرات,الشبكات والاتصالات


## 3) تنظيف البيانات

In [4]:
df = raw.copy()

# أسماء أعمدة إنجليزية لتسهيل العمل
df.columns = ['id', 'date', 'location', 'department', 'task', 'action', 'parts', 'category']

# إزالة المسافات الزائدة والتطويل (ـ) وتوحيد المسافات المكررة
text_cols = ['location', 'department', 'task', 'action', 'parts', 'category']
for c in text_cols:
    df[c] = (df[c].astype('string')
                  .str.replace('ـ', '', regex=False)
                  .str.replace(r'\s+', ' ', regex=True)
                  .str.strip()
                  .replace('', pd.NA))

# تحويل التاريخ (يوم/شهر/سنة)
df['date'] = pd.to_datetime(df['date'], format='%d/%m/%Y', errors='coerce')

# توحيد أسماء الأماكن المكتوبة بأكثر من صيغة
location_map = {
    'صاله انتظار عيادات الخارجية': 'صالة انتظار العيادات الخارجية',
    'البوابة الرئيسية للمستشفي': 'البوابة الرئيسية',
    'البدرم': 'البدروم',
    'غرف الكاميرات': 'غرفة الكاميرات',
    'غرفة المراقبة': 'غرفة الكاميرات',
    'وحدة تاهيل سمعي': 'وحدة التأهيل السمعي اللفظي',
    'وحدة التاهيل سمعي لفظي': 'وحدة التأهيل السمعي اللفظي',
    'وحدة التاهيل السمعي لفظي': 'وحدة التأهيل السمعي اللفظي',
    'السمع و التوازن': 'السمع والتوازن',
    'الصدلية الخارجية': 'الصيدلية الخارجية',
    'صيدلية المركزية': 'الصيدلية المركزية',
    'سكرتير البلع والتخاطب': 'سكرتير التخاطب والبلع',
}
df['location'] = df['location'].replace(location_map)

# "لا يوجد" في القطع = لم تُستخدم قطع
df['parts'] = df['parts'].replace('لا يوجد', pd.NA)

# فصل أيام الإجازة والصفوف الفارغة عن المهام الفعلية
df['is_leave'] = df['task'].eq('اجازة')
df['is_empty'] = df['task'].isna()
tasks = df[~df['is_leave'] & ~df['is_empty']].copy()

print('إجمالي الصفوف:', len(df))
print('أيام إجازة:', df['is_leave'].sum())
print('صفوف فارغة:', df['is_empty'].sum(), '→', df.loc[df['is_empty'], 'date'].dt.strftime('%d/%m').tolist())
print('مهام فعلية:', len(tasks))

إجمالي الصفوف: 50
أيام إجازة: 4
صفوف فارغة: 2 → ['27/10', '28/10']
مهام فعلية: 44


## 4) ملخص عام

In [5]:
work_days = tasks['date'].nunique()
summary = pd.Series({
    'عدد المهام': len(tasks),
    'أيام العمل التي سُجلت فيها مهام': work_days,
    'متوسط المهام في يوم العمل': round(len(tasks) / work_days, 2),
    'أيام الإجازة': int(df['is_leave'].sum()),
    'مهام استُخدمت فيها قطع': int(tasks['parts'].notna().sum()),
    'عدد الأقسام المستفيدة': tasks['department'].nunique(),
    'عدد الأماكن': tasks['location'].nunique(),
    'الفترة': f"{tasks['date'].min():%d/%m/%Y} – {tasks['date'].max():%d/%m/%Y}",
})
summary.to_frame('القيمة')

,القيمة
عدد المهام,44
أيام العمل التي سُجلت فيها مهام,15
متوسط المهام في يوم العمل,2.93
أيام الإجازة,4
مهام استُخدمت فيها قطع,8
عدد الأقسام المستفيدة,9
عدد الأماكن,19
الفترة,30/09/2026 – 26/10/2026


## 5) التوزيع حسب التصنيف الرئيسي

In [6]:
cat = tasks['category'].value_counts().rename_axis('التصنيف').reset_index(name='عدد المهام')
cat['النسبة %'] = (cat['عدد المهام'] / cat['عدد المهام'].sum() * 100).round(1)
display(cat)

fig = px.bar(cat, x='عدد المهام', y='التصنيف', orientation='h', text='عدد المهام',
             title='عدد المهام حسب التصنيف الرئيسي')
fig.update_layout(yaxis={'categoryorder': 'total ascending'}, title_x=0.5)
fig.show()

,التصنيف,عدد المهام,النسبة %
0,الشبكات والاتصالات,18,40.9
1,الدعم الفني والمستخدمين,11,25.0
2,إدارة الخوادم وأمن المعلومات,10,22.7
3,صيانة العتاد والملحقات,5,11.4


## 6) التوزيع حسب القسم المستفيد

In [7]:
dep = tasks['department'].value_counts().rename_axis('القسم').reset_index(name='عدد المهام')
display(dep)

fig = px.bar(dep, x='عدد المهام', y='القسم', orientation='h', text='عدد المهام',
             title='عدد المهام حسب القسم المستفيد')
fig.update_layout(yaxis={'categoryorder': 'total ascending'}, title_x=0.5)
fig.show()

,القسم,عدد المهام
0,قسم الخدمات المساندة,16
1,شعبة تكنولوجيا المعلومات,11
2,شعبة السمع والتوازن,9
3,شعبة الصيدلية,2
4,شعبة التخاطب والبلع,2
5,شعبة التمريض,1
6,قسم المالية,1
7,قسم العلاقات العامة,1
8,شعبة الأشعة,1


### القسم × التصنيف

In [8]:
ct = pd.crosstab(tasks['department'], tasks['category'], margins=True, margins_name='المجموع')
display(ct)

fig = px.imshow(pd.crosstab(tasks['department'], tasks['category']), text_auto=True,
                color_continuous_scale='Blues', aspect='auto',
                title='توزيع المهام: القسم × التصنيف')
fig.update_layout(title_x=0.5, xaxis_title='', yaxis_title='')
fig.show()

category,إدارة الخوادم وأمن المعلومات,الدعم الفني والمستخدمين,الشبكات والاتصالات,صيانة العتاد والملحقات,المجموع
department,,,,,
شعبة الأشعة,0,0,1,0,1
شعبة التخاطب والبلع,0,2,0,0,2
شعبة التمريض,0,0,0,1,1
شعبة السمع والتوازن,0,6,2,1,9
شعبة الصيدلية,0,0,2,0,2
شعبة تكنولوجيا المعلومات,9,1,0,1,11
قسم الخدمات المساندة,1,0,13,2,16
قسم العلاقات العامة,0,1,0,0,1
قسم المالية,0,1,0,0,1


## 7) الأماكن الأكثر طلبًا للخدمة

In [9]:
loc = tasks['location'].value_counts().rename_axis('المكان').reset_index(name='عدد المهام')
display(loc.head(10))

fig = px.bar(loc.head(10), x='عدد المهام', y='المكان', orientation='h', text='عدد المهام',
             title='أكثر 10 أماكن طلبًا للخدمة')
fig.update_layout(yaxis={'categoryorder': 'total ascending'}, title_x=0.5)
fig.show()

,المكان,عدد المهام
0,مكتب ICT,12
1,وحدة التأهيل السمعي اللفظي,7
2,غرفة الكاميرات,3
3,البوابة الرئيسية,3
4,الطابق الاول,2
5,سكرتير التخاطب والبلع,2
6,صالة انتظار العيادات الخارجية,2
7,السمع والتوازن,2
8,خلف غرفة الاجتماعات المركزية,1
9,لوب قسم السمع والتوازن,1


## 8) التحليل الزمني

In [10]:
daily = tasks.groupby('date').size().rename('عدد المهام').reset_index()
fig = px.bar(daily, x='date', y='عدد المهام', text='عدد المهام', title='عدد المهام اليومية')
leave_days = df.loc[df['is_leave'], 'date']
for d in leave_days:
    fig.add_vline(x=d, line_dash='dot', line_color='gray')
fig.update_layout(title_x=0.5, xaxis_title='التاريخ (الخطوط المنقطة = إجازة)')
fig.show()

# حسب يوم الأسبوع
ar_days = {'Saturday':'السبت','Sunday':'الأحد','Monday':'الإثنين','Tuesday':'الثلاثاء',
           'Wednesday':'الأربعاء','Thursday':'الخميس','Friday':'الجمعة'}
order = ['السبت','الأحد','الإثنين','الثلاثاء','الأربعاء','الخميس','الجمعة']
tasks['weekday'] = tasks['date'].dt.day_name().map(ar_days)
wd = tasks['weekday'].value_counts().reindex(order).dropna().astype(int)
display(wd.to_frame('عدد المهام'))

# حسب الأسبوع
weekly = tasks.groupby(tasks['date'].dt.to_period('W-FRI')).size()
weekly.index = [f"{p.start_time:%d/%m} – {p.end_time:%d/%m}" for p in weekly.index]
display(weekly.to_frame('عدد المهام'))

,عدد المهام
weekday,
الأحد,8
الإثنين,14
الثلاثاء,4
الأربعاء,11
الخميس,7


,عدد المهام
26/09 – 02/10,3
03/10 – 09/10,9
10/10 – 16/10,17
17/10 – 23/10,10
24/10 – 30/10,5


## 9) القطع المستخدمة

In [11]:
parts = tasks['parts'].dropna().value_counts().rename_axis('القطعة').reset_index(name='مرات الاستخدام')
display(parts)

fig = px.pie(parts, names='القطعة', values='مرات الاستخدام', hole=0.45, title='القطع المستخدمة')
fig.update_layout(title_x=0.5)
fig.show()

,القطعة,مرات الاستخدام
0,كابل كاميرات,3
1,Female rj45,2
2,BNC,2
3,علبة ضد الرطوبة,1


## 10) المواضيع المتكررة (بالكلمات المفتاحية)

In [12]:
topics = {
    'كاميرات المراقبة / DVR': r'كامير|DVR|cam-',
    'النسخ الاحتياطي والترحيل': r'احتياطي|ترحيل|backup|أرشف',
    'الشبكة اللاسلكية': r'لاسلكي|UNIFI',
    'البصمة / الموارد البشرية': r'بصم|الموارد البشرية',
    'الحسابات والصلاحيات': r'حساب|صلاحي|باسورد|البريد',
    'البرامج والأوفيس': r'برنامج|برامج|الأوفيس|PDF|FTP',
    'الدومين / المزامنة': r'دومين|دوميين|المزامن|التاريخ|الوقت',
    'الطابعات': r'طابع',
}
text = tasks['task'].fillna('') + ' ' + tasks['action'].fillna('')
topic_counts = pd.Series({k: text.str.contains(v, case=False, regex=True).sum() for k, v in topics.items()}) \
                 .sort_values(ascending=False)
display(topic_counts.to_frame('عدد المهام'))

fig = px.bar(topic_counts.sort_values(), orientation='h', text_auto=True,
             title='المواضيع المتكررة في المهام', labels={'value':'عدد المهام','index':''})
fig.update_layout(showlegend=False, title_x=0.5)
fig.show()

,عدد المهام
كاميرات المراقبة / DVR,15
النسخ الاحتياطي والترحيل,9
الحسابات والصلاحيات,7
البرامج والأوفيس,6
البصمة / الموارد البشرية,3
الشبكة اللاسلكية,3
الدومين / المزامنة,3
الطابعات,1


### تتبّع الكاميرات وأجهزة DVR التي تكررت أعطالها

In [13]:
cams = text.str.extractall(r'(cam-\d+)\s*/\s*(DVR-\d+)', flags=0)
if len(cams):
    cams.columns = ['الكاميرا', 'DVR']
    display(cams.value_counts().to_frame('مرات الظهور'))

cam_tasks = tasks[text.str.contains(topics['كاميرات المراقبة / DVR'], case=False, regex=True)]
cam_tasks[['date', 'location', 'task', 'action', 'parts']]

,,مرات الظهور
الكاميرا,DVR,
cam-01,DVR-03,1
cam-07,DVR-03,1
cam-12,DVR-01,1
cam-13,DVR-04,1


,date,location,task,action,parts
2,2026-10-01,خلف غرفة الاجتماعات المركزية,صيانة كاميرة المراقبة بجوار غرفة الاجتماعات وتغطي الساحة الخارجية للمستشفي,تغير Female rj45,Female rj45
4,2026-10-05,صالة انتظار العيادات الخارجية,تمديد كابل كاميرا جديد بدل الكابل القديم المتهالك cam-07 /DVR-03,تمديد كابل كاميرات,كابل كاميرات
5,2026-10-05,في طوابق المستشفي,تم معالجة مشكلة كاميرات cam-01/DVR-03 & cam-12/DVR-01,تغير BNC,BNC
6,2026-10-05,لوب قسم السمع والتوازن,تمديد كابل كاميرا جديد بدل الكابل القديم المتهالك cam-13 /DVR-04,تمديد كابل كاميرات,كابل كاميرات
8,2026-10-07,غرفة الكاميرات,تركيب استبدل جهاز DVR 8-Channal بجهاز DVR 16-Channal,تبديل جهاز,<NA>
9,2026-10-07,مكتب ICT,تم نقل الهارد القديم واضافة هارد جديد على DVR 16-channal مع الحفاظ على التسجيلات السابقة,نقل واضافة هارد دسك,<NA>
10,2026-10-07,الحديقة الخارجية,تركيب علبة حماية في المنهل في الحديقة الخارجية لحماية سوتش الخارجية المغذي للكاميرات الخارجية,تركيب علبة حماية ضد الرطوبة,علبة ضد الرطوبة
11,2026-10-08,غرفة الكاميرات,إعادة ضبط (Reset) كاميرا البدروم (تغطية المصاعد)&كاميرا مدخل الأطراف.&كاميرا المطبخ.,برمجة الكاميرات وتعريفها من جديد,<NA>
12,2026-10-08,صالة انتظار العيادات الخارجية,استبدال كابل التغذية القديم بكابل جديد لوجود عطل تسبب في انقطاع الكهرباء، وتم استعادة إشارة الكاميرا وإعادتها للعمل ...,تمديد كابل كاميرات,كابل كاميرات
13,2026-10-11,البدروم,معالجة كاميرا ممر مخزن المستهلكات,تغير BNC,BNC


## 11) المهام الروتينية المتكررة (نفس الوصف)

In [14]:
rep = tasks.groupby('task').agg(مرات=('id','size'), التواريخ=('date', lambda s: ', '.join(s.dt.strftime('%d/%m'))))
rep[rep['مرات'] > 1].sort_values('مرات', ascending=False)

,مرات,التواريخ
task,,
مراجعة نسخ الاحتياطي للانظمة والتاكد من عملية الترحيل مهمة روتنية,2,"11/10, 25/10"
مراجعة نسخ الاحتياطي لنظام المرضي والتاكد من عملية الترحيل من سيرفرات الوزارة الي سيرفرات المستشفي,2,"11/10, 25/10"


## 12) ملاحظات جودة البيانات

In [15]:
issues = []
issues.append(f"صفوف بلا أي بيانات: {df['is_empty'].sum()} ({', '.join(df.loc[df['is_empty'],'date'].dt.strftime('%d/%m'))})")
miss = tasks[['location','department','action','category']].isna().sum()
for c, n in miss.items():
    if n: issues.append(f"مهام بدون {c}: {n}")
# مهام صُنّفت "الشبكات" بينما الإجراء كهربائي/نقل
issues.append('بعض أسماء الأماكن كانت مكتوبة بصيغ مختلفة وتم توحيدها في الخطوة 3')
for i in issues: print('•', i)

• صفوف بلا أي بيانات: 2 (27/10, 28/10)
• بعض أسماء الأماكن كانت مكتوبة بصيغ مختلفة وتم توحيدها في الخطوة 3


## 13) تصدير النتائج إلى Excel في Drive

In [16]:
OUT = '/content/drive/MyDrive/ICT_10-2026_analysis.xlsx'
with pd.ExcelWriter(OUT) as w:
    summary.to_frame('القيمة').to_excel(w, sheet_name='ملخص')
    cat.to_excel(w, sheet_name='التصنيفات', index=False)
    dep.to_excel(w, sheet_name='الأقسام', index=False)
    loc.to_excel(w, sheet_name='الأماكن', index=False)
    ct.to_excel(w, sheet_name='قسم×تصنيف')
    parts.to_excel(w, sheet_name='القطع', index=False)
    topic_counts.to_frame('عدد المهام').to_excel(w, sheet_name='المواضيع')
    tasks.drop(columns=['is_leave','is_empty']).to_excel(w, sheet_name='البيانات النظيفة', index=False)
print('تم الحفظ في:', OUT)

تم الحفظ في: /content/drive/MyDrive/ICT_10-2026_analysis.xlsx
